# Your First IMF Series

Download a series from the [IMF Data API](https://data.imf.org) into Julia. The API is a standard SDMX 2.1 REST service and needs **no API key**. It returns CSV only when the request asks for it in the `Accept` header, and `CSV.jl` then reads the reply as a `Tables.jl` table.

In [ ]:
Pollis.packages("HTTP", "CSV", "Tables", "Plots")

In [ ]:
# import Pkg; Pkg.add(["HTTP", "CSV", "Tables", "Plots"])
using HTTP, CSV, Tables, Plots

base = "https://api.imf.org/external/sdmx/2.1/data"
csvheader = ["Accept" => "application/vnd.sdmx.data+csv;version=1.0.0"]
imf(flow, key; query = "") = CSV.File(HTTP.get("$base/$flow/$key?$query", csvheader).body)

## One Series from the World Economic Outlook

The dataflow is `IMF.RES,WEO`. Its key has three dimensions, `COUNTRY.INDICATOR.FREQUENCY`: here Portugal, real GDP growth in percent, annual. The World Economic Outlook includes forecasts; the column `LATEST_ACTUAL_ANNUAL_DATA` gives the last year of actual data.

In [ ]:
tbl = imf("IMF.RES,WEO", "PRT.NGDP_RPCH.A"; query = "startPeriod=2015")

years = Tables.getcolumn(tbl, :TIME_PERIOD)
growth = Tables.getcolumn(tbl, :OBS_VALUE)
actual = first(Tables.getcolumn(tbl, :LATEST_ACTUAL_ANNUAL_DATA))
foreach((y, g) -> println(y, "  ", round(g; digits = 2), y > actual ? "  (forecast)" : ""), years, growth)

## Several Countries in One Request

Join several values of one dimension with `+`. The reply has one row per country and year, and the `COUNTRY` column says which is which.

In [ ]:
tbl = imf("IMF.RES,WEO", "PRT+ESP+DEU.NGDP_RPCH.A"; query = "startPeriod=2015")

country = Tables.getcolumn(tbl, :COUNTRY)
years = Tables.getcolumn(tbl, :TIME_PERIOD)
growth = Tables.getcolumn(tbl, :OBS_VALUE)

p = plot(xlabel = "year", ylabel = "real GDP growth (%)", legend = :bottomleft)
for c in unique(country)
    rows = findall(==(c), country)
    plot!(p, years[rows], growth[rows], marker = :circle, label = c)
end
vline!(p, [actual + 0.5], linestyle = :dash, color = :gray, label = "forecasts to the right")
p

## Next Steps

- Change the indicator: `PCPIPCH` (inflation), `LUR` (unemployment), `GGXWDG_NGDP` (government debt, % of GDP).
- Try other dataflows such as `IMF.STA,CPI` or `IMF.STA,ER`; the panel's tabs show their keys.